# Stage 4: Model Training
Adds 6 interaction features, log transforms CLV, runs 5 fold CV to select max_depth, trains XGBoost with early stopping, saves artifacts to S3.

Input: s3://ins-churn-data/features/train.csv, val.csv

Output: s3://ins-churn-data/model_artifacts/xgb_model.json, scaler.pkl, feature_names.json, model_config.json

In [ ]:
import boto3, pandas as pd, numpy as np, io, os, json, logging, pickle
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
FEAT_DIR = 'features'
ARTIFACT_DIR = 'model_artifacts'
LOCAL_TMP = '/tmp/ins_churn'
os.makedirs(LOCAL_TMP, exist_ok=True)

s3 = boto3.client('s3', region_name=REGION)


def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)


def write_json_to_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key,
                  Body=json.dumps(data, indent=2, default=str))


def upload_file_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)
    log.info('  uploaded to s3://%s/%s', bucket, key)


log.info('Imports ready.')

In [ ]:
def add_interaction_features(df):
    """Add 6 engineered interaction terms.

    Uses a safe_col helper so missing columns produce 0 instead of KeyError.
    """
    out = df.copy()

    def safe_col(name):
        return out[name] if name in out.columns else pd.Series(0, index=out.index)

    # 1. Premium times tenure: high value long term customers
    out['premium_x_tenure'] = (
        safe_col('premium_amount') * safe_col('policy_tenure_months')
    )

    # 2. Delay times complaints: compounding churn signal
    out['delay_x_complaints'] = (
        safe_col('payment_delay_days') * safe_col('customer_complaints')
    )

    # 3. Claims times premium: high claim high premium is unprofitable
    out['claims_x_premium'] = (
        safe_col('claim_frequency') * safe_col('premium_amount')
    )

    # 4. Margin times tenure: long retained profitable policy
    out['margin_x_tenure'] = (
        safe_col('policy_margin') * safe_col('policy_tenure_months')
    )

    # 5. Agent tenure times premium: experienced agents win bigger policies
    out['agent_tenure_x_premium'] = (
        safe_col('agent_tenure_days') * safe_col('premium_amount')
    )

    # 6. Customer tenure times margin pct: loyal customers with healthy margin
    out['cust_tenure_x_margin_pct'] = (
        safe_col('customer_tenure_days') * safe_col('margin_pct')
    )

    return out


log.info('add_interaction_features defined.')

In [ ]:
TARGET_COL = 'customer_lifetime_value'


def prepare_xy(df, feature_names=None, scaler=None, fit_scaler=False):
    """Extract X and y, apply or fit scaler, log transform target.

    Returns X_scaled, feature_names, y_raw, y_log, scaler.
    X_scaled is the scaled feature matrix.
    y_raw is the original CLV values, used for metric reporting.
    y_log is log1p(CLV), used for training.
    """
    # Align to known feature list if provided
    if feature_names is not None:
        cols = feature_names + ([TARGET_COL] if TARGET_COL in df.columns else [])
        df = df.reindex(columns=cols, fill_value=0)

    y_raw = df[TARGET_COL].values if TARGET_COL in df.columns else None
    y_log = np.log1p(y_raw) if y_raw is not None else None

    feat_cols = [c for c in df.columns if c != TARGET_COL]
    X = df[feat_cols].values.astype(np.float64)

    if fit_scaler:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    elif scaler is not None:
        X_scaled = scaler.transform(X)
    else:
        X_scaled = X

    return X_scaled, feat_cols, y_raw, y_log, scaler


log.info('prepare_xy defined.')

In [ ]:
# Load train and val
train = read_csv_from_s3(BUCKET_NAME, f'{FEAT_DIR}/train.csv')
val = read_csv_from_s3(BUCKET_NAME, f'{FEAT_DIR}/val.csv')

train_e = add_interaction_features(train)
val_e = add_interaction_features(val)

X_tr, feature_names, y_tr_raw, y_tr_log, scaler = prepare_xy(train_e, fit_scaler=True)
X_vl, _, y_vl_raw, y_vl_log, _ = prepare_xy(val_e, feature_names=feature_names, scaler=scaler)

log.info('X_tr: %s, X_vl: %s', X_tr.shape, X_vl.shape)
log.info('y_log range: %.2f to %.2f, std: %.2f', y_tr_log.min(), y_tr_log.max(), y_tr_log.std())

In [ ]:
# 5 fold CV to select best max_depth
DEPTH_CANDIDATES = [3, 4, 5, 6]
cv_results = {}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for depth in DEPTH_CANDIDATES:
    candidate = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=depth,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
    scores = cross_val_score(candidate, X_tr, y_tr_log,
                             cv=kf, scoring='r2', n_jobs=-1)
    cv_results[depth] = scores.mean()
    log.info('  max_depth=%d  CV R2=%.4f (std %.4f)', depth, scores.mean(), scores.std())

best_depth = max(cv_results, key=cv_results.get)
log.info('Best max_depth: %d  (CV R2=%.4f)', best_depth, cv_results[best_depth])

In [ ]:
# Final training with early stopping
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=best_depth,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric='rmse',
)

model.fit(
    X_tr, y_tr_log,
    eval_set=[(X_vl, y_vl_log)],
    verbose=100,
)

log.info('Best iteration: %d', model.best_iteration)

In [ ]:
# Quick validation sanity check
preds_log = model.predict(X_vl)
preds_raw = np.expm1(preds_log)
residuals = y_vl_raw - preds_raw

mae = mean_absolute_error(y_vl_raw, preds_raw)
rmse = np.sqrt(mean_squared_error(y_vl_raw, preds_raw))
r2 = r2_score(y_vl_raw, preds_raw)
mape = np.mean(np.abs(residuals / np.clip(y_vl_raw, 1, None))) * 100

log.info('Val  R2=%.4f  RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%', r2, rmse, mae, mape)
if r2 >= 0.70 and rmse <= 5000:
    log.info('Gate: PASS')
else:
    log.info('Gate: FAIL, review CLV rebuild')

In [ ]:
# Save artifacts to S3 via /tmp

# XGBoost model (requires file path)
xgb_local = os.path.join(LOCAL_TMP, 'xgb_model.json')
model.save_model(xgb_local)
upload_file_to_s3(xgb_local, BUCKET_NAME, f'{ARTIFACT_DIR}/xgb_model.json')

# Scaler (pickle bytes, no file needed)
s3.put_object(Bucket=BUCKET_NAME,
              Key=f'{ARTIFACT_DIR}/scaler.pkl',
              Body=pickle.dumps(scaler))
log.info('  saved to s3://%s/%s/scaler.pkl', BUCKET_NAME, ARTIFACT_DIR)

# Feature names
write_json_to_s3(feature_names, BUCKET_NAME, f'{ARTIFACT_DIR}/feature_names.json')

# Model config
write_json_to_s3({'best_depth': best_depth,
                  'best_iteration': int(model.best_iteration),
                  'n_features': len(feature_names)},
                 BUCKET_NAME, f'{ARTIFACT_DIR}/model_config.json')

log.info('All model artifacts saved to S3.')